# GPU CONFIG

In [5]:
import os
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Running on:", DEVICE)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

Running on: cuda


# Load Books

In [6]:
import os
import re


def clean_text(text):

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Fix broken line breaks
    text = text.replace("\n", " ")

    # Remove weird control chars only
    text = re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", "", text)
    text = text.replace('\ufeff', '')

    return text.strip()


def load_all_books(folder_path):

    books = []
    files = sorted(os.listdir(folder_path))

    for idx, file in enumerate(files):

        if file.endswith(".txt"):

            path = os.path.join(folder_path, file)

            with open(path, "r", encoding="utf-8", errors="ignore") as f:

                raw_text = f.read()

                # cleaning books
                text = clean_text(raw_text)

                books.append({
                    "book_id": idx + 1,
                    "filename": file,
                    "text": text
                })

                print(f"Loaded: {file}")

    return books


books = load_all_books(
    "~/My Learning/projects/data-GOT"
)

Loaded: 001ssb.txt
Loaded: 002ssb.txt
Loaded: 003ssb.txt
Loaded: 004ssb.txt
Loaded: 005ssb.txt


# IMPORTS

In [ ]:
import os
import numpy as np
import pickle
import torch
import nltk
import faiss
import anthropic

from tqdm import tqdm
from nltk.tokenize import sent_tokenize

from transformers import pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder

nltk.download("punkt")

# Tokenizer

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
"sentence-transformers/all-mpnet-base-v2"
)

# Sentence Based Chunking (Context Preserving) with metadata (per book)

In [9]:
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer
import re


def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\n", " ").strip()
    return text


# Use the same tokenizer as the embedding model so token counts are accurate
_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-mpnet-base-v2")


def count_tokens(text):
    """Count tokens exactly as MPNet sees them — not word count."""
    return len(_tokenizer.encode(text, add_special_tokens=False))


# Change 7: token-based chunking replaces word-based (max_words=140)
# MPNet has a hard 512-token limit. Using max_tokens=400 stays safely under it.
# Previously 140 words could be 200+ tokens for GOT names (e.g. "Daenerys" = 3 tokens).
def sentence_chunker_with_meta(
    text,
    book_id,
    max_tokens=400,      # was: max_words=140
    overlap_tokens=80    # was: overlap_words=40
):
    text = clean_text(text)
    sentences = sent_tokenize(text)

    chunks = []
    current_sents = []
    current_token_count = 0
    chunk_id = 0
    global_pos = 0

    for sent in sentences:

        sent = clean_text(sent)
        sent_tokens = count_tokens(sent)

        if current_token_count + sent_tokens > max_tokens and current_sents:

            chunk_text = " ".join(current_sents)

            chunks.append({
                "chunk_uid": f"B{book_id}_C{chunk_id}",
                "text": chunk_text,
                "book": book_id,
                "chunk_id": chunk_id,
                "position": global_pos
            })

            global_pos += 1
            chunk_id += 1

            # Build overlap by walking back through sentences until
            # we hit the overlap_tokens budget
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current_sents):
                s_tok = count_tokens(s)
                if overlap_count + s_tok <= overlap_tokens:
                    overlap_sents.insert(0, s)
                    overlap_count += s_tok
                else:
                    break

            current_sents = overlap_sents + [sent]
            current_token_count = overlap_count + sent_tokens

        else:
            current_sents.append(sent)
            current_token_count += sent_tokens

    # Flush last chunk
    if current_sents:
        chunk_text = " ".join(current_sents)
        chunks.append({
            "chunk_uid": f"B{book_id}_C{chunk_id}",
            "text": chunk_text,
            "book": book_id,
            "chunk_id": chunk_id,
            "position": global_pos
        })

    return chunks


# ── NOTE ────────────────────────────────────────────────────────────────────
# Change 7 modifies how chunks are built.
# The existing embeddings_0.npy and chunk_metadata.pkl were built with
# the OLD word-based chunker (max_words=140).
#
# To apply this change fully you must re-run:
#   1. This cell (chunker definition)
#   2. The "Build chunks" cell (all_chunks = ...)
#   3. The "Batch Encode + Save Embeddings" cell (embed_and_save)
#   4. The "Save metadata" cell (pickle.dump)
#   5. The "Load Embeddings + Build FAISS Index" cell
#
# Estimated time: ~5 minutes on GPU.
# Until you re-run those cells, the system continues using the old chunks.
# ────────────────────────────────────────────────────────────────────────────

In [10]:
all_chunks = []

for book in books:

    book_chunks = sentence_chunker_with_meta(
        book["text"],
        book["book_id"]
    )

    all_chunks.extend(book_chunks)

print("Total chunks:", len(all_chunks))
# Prepare text for embedding
texts_for_embedding = [c["text"] for c in all_chunks]

Token indices sequence length is longer than the specified maximum sequence length for this model (895 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 7394


# Load Embedding Model (GPU + FP16)

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

if device == "cuda":
    model = model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# Embed Only the Text (Keep Metadata)
texts_for_embedding = [c["text"] for c in all_chunks]

# Batch Encode + Save Embeddings (Long Job)

In [13]:
def embed_and_save(
    texts,
    batch_size=128,
    save_every=50000,
    out_dir="embeddings"
):

    os.makedirs(out_dir, exist_ok=True)

    buffer = []
    total = 0
    file_id = 0

    for i in tqdm(range(0, len(texts), batch_size)):

        batch = texts[i:i+batch_size]

        embeddings = model.encode(
            batch,
            batch_size=batch_size,
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype("float32")

        buffer.append(embeddings)
        total += len(embeddings)

        if total >= save_every:

            arr = np.vstack(buffer)

            fname = f"{out_dir}/embeddings_{file_id}.npy"
            np.save(fname, arr)

            print(f"Saved: {fname}")

            buffer = []
            total = 0
            file_id += 1

    if buffer:

        arr = np.vstack(buffer)

        fname = f"{out_dir}/embeddings_{file_id}.npy"
        np.save(fname, arr)

        print(f"Saved final: {fname}")

In [14]:
# Run once to generate the embeddings
embed_and_save(texts_for_embedding)

100%|███████████████████████████████████████████████████████████████████████████████████| 58/58 [01:20<00:00,  1.38s/it]

Saved final: embeddings/embeddings_0.npy


In [15]:
with open("chunk_metadata.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

# load Embeddings + Build Faiss Index

In [16]:
def load_embeddings(folder):

    arrays = []

    for f in sorted(os.listdir(folder)):

        if f.endswith(".npy"):
            arr = np.load(os.path.join(folder, f))
            arrays.append(arr)

    return np.vstack(arrays)


print("Loading embeddings...")

# Convert to float32 (IMPORTANT)
embeddings = load_embeddings("embeddings").astype("float32")


# Build Faiss Index

dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)   # Cosine similarity
faiss.normalize_L2(embeddings)

index.add(embeddings)

print("FAISS index size:", index.ntotal)

Loading embeddings...
FAISS index size: 7394


In [17]:
with open("chunk_metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

# Fix: use metadata as all_chunks so the system works even if
# the chunking cells were not re-run after a kernel restart
all_chunks = metadata

print(f"Loaded {len(all_chunks)} chunks from metadata.")

Loaded 7394 chunks from metadata.


# Load Cross-Encoder Re-Ranker

In [18]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Load QA Model — Claude Haiku (replaces roberta-large-squad2)
# Why: RoBERTa has a 512-token hard limit — extra retrieved context is silently truncated.
# Claude Haiku handles 200K tokens, so all 27 retrieved chunks are actually used.

In [ ]:
# Claude Haiku replaces roberta-large-squad2 as the QA reader.
# Set your API key below or export ANTHROPIC_API_KEY in your environment.
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_API_KEY_HERE")

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
CLAUDE_MODEL = "claude-haiku-4-5-20251001"

print(f"Claude client ready — model: {CLAUDE_MODEL}")

# Question Search (Retriever + Reranker)

In [20]:
def search(question, top_k=30, expand_window=4):

    # Encode query
    q_emb = model.encode([question], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)

    scores, ids = index.search(q_emb, top_k)

    # Get full chunks from FAISS results
    candidates = [all_chunks[i] for i in ids[0]]

    # Rerank: CrossEncoder reads (question, chunk) together — more accurate
    pairs = [(question, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)

    reranked = list(zip(candidates, rerank_scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    # Change 3: use top-3 reranked chunks (not just rank-1)
    # Each contributes its window of neighbors for broader context
    seen_uids = set()
    window_chunks = []

    for chunk, score in reranked[:3]:
        for c in all_chunks:
            if (
                c["book"] == chunk["book"] and
                abs(c["position"] - chunk["position"]) <= expand_window and  # Change 2: window=4
                c["chunk_uid"] not in seen_uids
            ):
                window_chunks.append(c)
                seen_uids.add(c["chunk_uid"])

    # Sort by book then position for coherent reading order
    window_chunks.sort(key=lambda x: (x["book"], x["position"]))

    return window_chunks

# Answer Engine

In [ ]:
def answer_question(question):

    chunks = search(question)

    # Label each chunk with its source so Claude can reference them
    merged_context = "\n\n".join([
        f"[Book {c['book']}, chunk {c['chunk_id']}]\n{c['text']}"
        for c in chunks
    ])

    system_prompt = (
        "You are a question-answering assistant for the Game of Thrones / "
        "A Song of Ice and Fire book series. "
        "Answer questions using ONLY the provided context passages. "
        "If the answer is not in the context, reply with exactly: I don't know. "
        "Keep answers concise — one to three sentences."
    )

    user_prompt = (
        f"Context from the books:\n{merged_context}\n\n"
        f"Question: {question}\n\n"
        "Answer based only on the context above:"
    )

    response = claude_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=200,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )

    answer = response.content[0].text.strip()

    # Treat "I don't know" variants as a no-answer signal
    if answer.lower().startswith("i don't know") or answer.lower().startswith("i do not know"):
        return "I don't know", 0.0, "N/A"

    # Source attribution — which books / chunks contributed
    seen_books = set()
    sources = []
    for c in chunks[:3]:
        if c["book"] not in seen_books:
            sources.append(f"Book {c['book']} (chunk {c['chunk_id']})")
            seen_books.add(c["book"])
    source_str = " | ".join(sources) if sources else "N/A"

    # Claude doesn't return a numeric confidence score; use 1.0 as a placeholder
    return answer, 1.0, source_str

# Debug

In [24]:
def debug_context(question):

    chunks = search(question)

    print("\n==== CONTEXT ====\n")

    for i, c in enumerate(chunks):
        print(f"[{i}] Book {c['book']} Pos {c['position']}")
        print(c["text"][:300])
        print("-----")

In [25]:
ans, score, source = answer_question("Who were the siblings of robb stark?")
print("Answer    :", ans)
print("Confidence:", round(score, 3))
print("Source    :", source)

Answer    : SER HORAS and SER HOBBER
Confidence: 0.85
Source    : Book 2 (chunk 1369)


In [22]:
def run_cli():

    print("\nGOT Question Answering System")
    print("Type 'exit' to quit\n")

    while True:

        q = input("Ask> ")

        if q.lower() in ["exit", "quit"]:
            break

        ans, score, source = answer_question(q)

        print("\nAnswer    :", ans)
        print("Confidence:", round(score, 3))
        print("Source    :", source)
        print("-" * 50)

In [23]:
run_cli()


GOT Question Answering System
Type 'exit' to quit



Ask>  who killed ned stark?



Answer    : I don't know
Confidence: 0.203
Source    : N/A
--------------------------------------------------


Ask>  exit
